# Solution 04 — consumer groups & manual offsets

Reference for the manual-commit pattern and the two-consumer experiment. The interesting bit isn't the code; it's the *behaviour* you should observe.

In [ ]:
from confluent_kafka import Consumer

consumer = Consumer({
    'bootstrap.servers':  'redpanda:29092',
    'group.id':            'manual-commit-demo',
    'auto.offset.reset':   'earliest',
    'enable.auto.commit':  False,
})
consumer.subscribe(['strom'])

messages_read, empty_polls = 0, 0
while messages_read < 5 and empty_polls < 5:
    msg = consumer.poll(2.0)
    if msg is None: empty_polls += 1; continue
    if msg.error(): continue
    empty_polls = 0; messages_read += 1
    print(f'#{messages_read} offset={msg.offset()} -- processing...', end='')
    consumer.commit(message=msg)
    print(' committed.')
consumer.close()

## Task A — observing re-delivery without commit

We deliberately *don't* commit. Run this twice in a row with the same `group.id` and you'll see the same offsets pop up both times.

In [ ]:
from confluent_kafka import Consumer
consumer = Consumer({
    'bootstrap.servers':  'redpanda:29092',
    'group.id':            'no-commit-demo',
    'auto.offset.reset':   'earliest',
    'enable.auto.commit':  False,
})
consumer.subscribe(['strom'])
for i in range(3):
    msg = consumer.poll(2.0)
    if msg and not msg.error():
        print(f'#{i+1} offset={msg.offset()} -- (NOT committing)')
consumer.close()

## Task B — two consumers in one group

In [ ]:
from confluent_kafka import Consumer
from datetime import datetime
consumer = Consumer({
    'bootstrap.servers': 'redpanda:29092',
    'group.id':          'group-experiment',
    'auto.offset.reset': 'latest',
})
consumer.subscribe(['strom', 'wasser'])
try:
    while True:
        msg = consumer.poll(0.5)
        if msg and not msg.error():
            ts = datetime.now().strftime('%H:%M:%S')
            print(f'[{ts}] {msg.topic()} P{msg.partition()} '
                  f'offset={msg.offset()}')
except KeyboardInterrupt:
    consumer.close()